# Python language basics for data work

## Learning objectives

By the end of this notebook you will be able to:

- name the common Python types and check them with `type`;
- choose between a `list`, `tuple`, `set`, and `dict`, and explain the trade-off;
- write `if` / `elif` / `else` branches and build a value from them;
- loop with `for`, `while`, `enumerate`, and `zip`, and accumulate results by hand;
- handle errors deliberately with `try` / `except` instead of letting a script crash.

## Concept

Before reaching for a library, it helps to be fluent in the small set of Python features that
appear in every data script: values and types, the four built-in containers, branching, loops,
and error handling. This notebook practices each one on a real table so the examples are not
abstract.

Python values carry a **type**. Numbers are `int` or `float`, text is `str`, truth values are
`bool`, and "no value" is `None`. A **container** holds other values. The four built-ins cover
almost every need:

| Container | Ordered | Mutable | Duplicates | Typical use |
|---|---|---|---|---|
| `list` | yes | yes | yes | a sequence you will grow or sort |
| `tuple` | yes | no | yes | a fixed record, safe to share |
| `set` | no | yes | no | membership tests and de-duplication |
| `dict` | insertion order | yes | unique keys | looking a value up by a key |

**Conditionals** choose one branch based on a comparison. **Loops** repeat work. Python's `for`
iterates over the items of any container, while `while` repeats until a condition changes.

**Exceptions** are how Python reports failure. Rather than checking every possible problem in
advance, idiomatic Python tries an operation and catches the specific error it can recover from.
Catching a broad `Exception` hides bugs; catching the narrow error you expected keeps them
visible.

## Worked example

We start by loading the penguins table through the shared package. Everything below works on
plain Python containers, so we deliberately pull a few columns out as lists.

### Setup and types

In [ ]:
from ds_practice import load_penguins

penguins = load_penguins()
print("rows, columns:", penguins.shape)
penguins.head()

In [ ]:
masses = penguins["body_mass_g"].dropna().tolist()
species = penguins["species"].tolist()
islands = penguins["island"].tolist()

print("first mass:", masses[0], "->", type(masses[0]).__name__)
print("first species:", species[0], "->", type(species[0]).__name__)
print("column types:")
print(penguins.dtypes.to_dict())

### The four containers

Each container below is built from the same underlying column, which makes the differences
easy to see: a list keeps order and repeats, a set collapses to distinct values, a tuple pins
two numbers together, and a dict maps a key to a count.

In [ ]:
# list: ordered, mutable, repeats allowed
first_five = masses[:5]
first_five.append(4000)
print("list  :", first_five)

# tuple: fixed pair of summary values, cannot be reassigned
mass_range = (min(masses), max(masses))
print("tuple :", mass_range, "min-max span =", mass_range[1] - mass_range[0])

# set: distinct values only
print("set   :", sorted(set(species)))
print("is 'Gentoo' present?", "Gentoo" in set(species))

# dict: count penguins per species with a loop
counts = {}
for name in species:
    counts[name] = counts.get(name, 0) + 1
print("dict  :", counts)

`dict.get(key, default)` avoids a special case for the first occurrence: if the key is missing it
returns `0`, then we add one. The same pattern works for any tally.

### Conditionals

A chain of comparisons turns a number into a label. The boundaries below match the tiny
`mass_band` helper packaged in `pyutils.py`, so we can reuse the idea later.

In [ ]:
def coarse_band(grams):
    if grams < 3500:
        return "light"
    elif grams < 4500:
        return "medium"
    else:
        return "heavy"

for grams in [3200, 3500, 4499, 4500, 5100]:
    print(grams, "->", coarse_band(grams))

### Loops

A `for` loop walks the items directly. When the index matters, `enumerate` supplies it, and
`zip` pairs two sequences position by position.

In [ ]:
# count species with an explicit loop and accumulate a running total
running = 0
for i, grams in enumerate(masses[:5]):
    running += grams
    print(f"step {i}: mass={grams:>4} g, running total={running:>5} g")

# zip the island and species columns together for the first three rows
print()
for island, name in zip(islands[:3], species[:3]):
    print(f"{name:<10} lives on {island}")

# while: keep halving a mass until it drops below 1 kg
value = masses[0]
steps = 0
while value >= 1000:
    value = value / 2
    steps += 1
print(f"\nhalved {masses[0]} g below 1 kg in {steps} steps (final {value:.0f} g)")

### Exceptions

Two common failure modes: dividing by zero and reading a key that is absent. Each gets a narrow
`except` clause so unrelated problems still surface.

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float("nan")

def lookup(mapping, key):
    try:
        return mapping[key]
    except KeyError:
        return None

print("10 / 4 =", safe_divide(10, 4))
print("10 / 0 =", safe_divide(10, 0))
print("counts.get fallback:", lookup(counts, "Adelie"))
print("missing key        :", lookup(counts, "Emperor"))

Notice that `safe_divide(10, 0)` returns `nan` (not a number) rather than stopping the notebook.
In a real analysis you would decide whether `nan` is the right answer or whether the row should
be dropped.

## Exercises

Try these on the loaded `penguins` frame. Write your work in new cells.

1. **Count per island with a loop.** Build a `dict` that maps each island name to the number of
   penguins on it, using a plain `for` loop and `.get`. Do not use pandas.
2. **De-duplicate and range.** Produce a `set` of distinct `species`, then a `tuple` holding the
   minimum and maximum `flipper_length_mm`. Print both.
3. **Fail gracefully.** Write a function `safe_mean(values)` that returns `None` for an empty
   sequence and the arithmetic mean otherwise. Test it on `[]` and on the first ten masses.

## Limitations

The examples use small Python loops for teaching, but those loops are the slow path. Once the
data grows, the vectorised pandas and NumPy versions in notebook 04 replace them and run orders
of magnitude faster. The error handling here shows only `ZeroDivisionError` and `KeyError`; real
programs also contend with `ValueError`, `TypeError`, and I/O errors, which notebook 03 covers.
Finally, `float("nan")` is a sentinel that silently propagates through arithmetic, so it is a
convenient teaching device but a poor default for production code.